# Spotify LLM plot 생성

Kaggle refine CSV의 audio feature를 LLM에 보내 분위기 설명을 받고, `plot` 컬럼으로 저장.

In [4]:
import os
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

PROJECT_DIR = Path(".").resolve()
DATA_DIR = PROJECT_DIR / "raw_data"
REPO_ROOT = PROJECT_DIR.parent
ENV_FILE = REPO_ROOT / ".env"

INPUT_CSV = DATA_DIR / "spotify_kaggle_refine.csv"
OUTPUT_CSV = DATA_DIR / "spotify_llm.csv"

LLM_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
LLM_SLEEP_SEC = 0.3
SAVE_EVERY = 20

PITCH_CLASSES = ("C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B")

SYSTEM_PROMPT = (
    "당신은 음악 큐레이터입니다. Spotify 오디오 특성 수치를 반영해 "
    "곡의 분위기를 한국어로 묘사합니다. 수치를 그대로 나열하지 않고, "
    "곡마다 구별되게 2~3문장으로 작성합니다. "
    "가사·줄거리·제목에서 추측하지 말고, 주어진 오디오 특성과 장르만으로 묘사합니다.\n\n"
    "수치 스케일 (Spotify 공식):\n"
    "- energy, valence, danceability, acousticness: 각각 0.0~1.0 (1에 가까울수록 높음)\n"
    "- tempo: BPM (숫자가 클수록 빠름)\n"
    "- loudness: dB (0에 가까울수록 큼, 보통 -60~0)\n"
    "- mode: major(장조) / minor(단조)\n"
    "- key: C~B 음이름"
)

USER_PROMPT = """다음 곡의 분위기를 설명하세요.

제목: {title}
아티스트: {artist}
장르: {genre}
energy: {energy}
valence: {valence}
danceability: {danceability}
tempo: {tempo}
acousticness: {acousticness}
loudness: {loudness}
mode: {mode}
key: {key}
duration_ms: {duration_ms}"""

In [5]:
def build_user_prompt(row: pd.Series) -> str:
    mode = "major" if int(row["mode"]) == 1 else "minor"
    key = PITCH_CLASSES[int(row["key"]) % 12]

    return USER_PROMPT.format(
        title=row["track_name"],
        artist=row["track_artist"],
        genre=row["genre"],
        energy=row["energy"],
        valence=row["valence"],
        danceability=row["danceability"],
        tempo=row["tempo"],
        acousticness=row["acousticness"],
        loudness=row["loudness"],
        mode=mode,
        key=key,
        duration_ms=int(row["duration_ms"]),
    )


def llm_client() -> OpenAI:
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError("OPENAI_API_KEY 가 .env 에 없습니다.")
    return OpenAI(api_key=api_key)


def request_plot(prompt: str, client: OpenAI) -> str:
    resp = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=0.2,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
    )
    return resp.choices[0].message.content.strip()


def fill_plot_column(df: pd.DataFrame, client: OpenAI) -> pd.DataFrame:
    out = df.copy()
    if "plot" not in out.columns:
        out["plot"] = ""

    done = 0
    for i, (idx, row) in enumerate(out.iterrows(), start=1):
        if out.at[idx, "plot"]:
            continue

        out.at[idx, "plot"] = request_plot(build_user_prompt(row), client)
        done += 1

        if done % SAVE_EVERY == 0:
            out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
            print(f"[save] {done}건 생성, {OUTPUT_CSV.name}")
        time.sleep(LLM_SLEEP_SEC)

        if i % 50 == 0:
            print(f"[progress] {i}/{len(out)}")

    return out

In [7]:
load_dotenv(ENV_FILE)

df = pd.read_csv(INPUT_CSV)
if OUTPUT_CSV.exists():
    cached = pd.read_csv(OUTPUT_CSV)
    plot_map = {
        row["track_id"]: row["plot"]
        for _, row in cached.iterrows()
        if row.get("plot")
    }
    df["plot"] = df["track_id"].map(plot_map).fillna("")
    print(f"[resume] 기존 plot {df['plot'].ne('').sum()}건 로드")
else:   
    df["plot"] = ""

print(f"대상 {len(df)}건, plot 없음 {(df['plot'] == '').sum()}건")

client = llm_client()
df = fill_plot_column(df, client)
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"[done] {OUTPUT_CSV}")

대상 1437건, plot 없음 1437건
[save] 20건 생성, spotify_llm.csv
[save] 40건 생성, spotify_llm.csv
[progress] 50/1437
[save] 60건 생성, spotify_llm.csv
[save] 80건 생성, spotify_llm.csv
[save] 100건 생성, spotify_llm.csv
[progress] 100/1437
[save] 120건 생성, spotify_llm.csv
[save] 140건 생성, spotify_llm.csv
[progress] 150/1437
[save] 160건 생성, spotify_llm.csv
[save] 180건 생성, spotify_llm.csv
[save] 200건 생성, spotify_llm.csv
[progress] 200/1437
[save] 220건 생성, spotify_llm.csv
[save] 240건 생성, spotify_llm.csv
[progress] 250/1437
[save] 260건 생성, spotify_llm.csv
[save] 280건 생성, spotify_llm.csv
[save] 300건 생성, spotify_llm.csv
[progress] 300/1437
[save] 320건 생성, spotify_llm.csv
[save] 340건 생성, spotify_llm.csv
[progress] 350/1437
[save] 360건 생성, spotify_llm.csv
[save] 380건 생성, spotify_llm.csv
[save] 400건 생성, spotify_llm.csv
[progress] 400/1437
[save] 420건 생성, spotify_llm.csv
[save] 440건 생성, spotify_llm.csv
[progress] 450/1437
[save] 460건 생성, spotify_llm.csv
[save] 480건 생성, spotify_llm.csv
[save] 500건 생성, spotify_llm.csv
[